# 05 · Integración de resultados

Une los resultados parciales de los notebooks 01–04 en `results/resultados.json` y genera `results/RESUMEN_RESULTADOS.md`.
**Toda cifra del paper y de la presentación debe salir de estos dos archivos.**

In [1]:
%run ./00_configuracion.ipynb

In [2]:
def write_summary(R):
    """results/RESUMEN_RESULTADOS.md: cifras clave para el paper y la presentación."""
    f = lambda x, d=3: f"{x:.{d}f}".replace(".", ",")
    D, E, P, T = R["datos"], R["modelo_explicativo"], R["prediccion"], R["tamano_muestra"]
    d = P["desempeno"]
    def orr(model, term):
        o = E[model][term]
        return f"OR {f(o['OR'], 2)} (IC 95 % {f(o['IC95_inf'], 2)}–{f(o['IC95_sup'], 2)}; p {fmt_p(o['p']) if o['p'] < 0.001 else '= ' + fmt_p(o['p'])})"
    sig = lambda model: [term_label(t) for t, o in E[model].items() if o["p"] < ALPHA]
    nb = lambda k: P["comparaciones"][k]["nadeau_bengio"]
    L = ["# Resumen de resultados", "",
         "> Generado automáticamente por `make all`. No editar a mano: toda cifra del paper y de la presentación debe salir de aquí o de `resultados.json`.", "",
         "## Datos", "",
         f"- Fuente: {D['fuente']}. SHA-256 verificado.",
         f"- {D['n_total']} registros; {D['n_total'] - D['n_completos']} con faltantes (ca: {D['faltantes'].get('ca', 0)}, thal: {D['faltantes'].get('thal', 0)}); muestra analítica n = {D['n_completos']}, con {D['eventos_cc']} eventos ({f(100 * D['eventos_cc'] / D['n_completos'], 1)} %).",
         f"- Auditoría Kaggle: target invertido = {D['auditoria_kaggle']['target_invertido']}; duplicados = {D['auditoria_kaggle']['duplicados_kaggle']}; faltantes ocultos = {D['auditoria_kaggle']['faltantes_ocultos_kaggle']}.", "",
         "## Asociación (OE2, H-1)", "",
         f"- Modelo básico: LR χ² = {f(E['basico']['LR_chi2'], 1)}, gl = {E['basico']['LR_gl']}, p < 0,001 → **se rechaza H0 de H-1**. R² de Nagelkerke = {f(E['basico']['R2_Nagelkerke'])}; Hosmer-Lemeshow p = {f(E['basico']['Hosmer_Lemeshow_p'])}; VIF máximo = {f(E['basico']['VIF_max'], 2)}; Box-Tidwell p mínimo = {f(min(E['basico']['Box_Tidwell_p'].values()))}.",
         "- Variables significativas en el modelo básico: " + "; ".join(sig("OR_basico")) + ".",
         "- Variables significativas en el modelo extendido: " + "; ".join(sig("OR_extendido")) + ".",
         f"- Dolor asintomático (básico): {orr('OR_basico', 'cp=4')}. Sexo masculino: {orr('OR_basico', 'sex')}.",
         f"- FC máxima por 10 lpm (básico): {orr('OR_basico', 'thalach')}. Depresión ST por mm: {orr('OR_basico', 'oldpeak')}.",
         f"- Colesterol por 40 mg/dL (básico): {orr('OR_basico', 'chol')}. Edad por 10 años: {orr('OR_basico', 'age')}.",
         f"- Extendido: vasos por fluoroscopia {orr('OR_extendido', 'ca')}; talio con defecto reversible {orr('OR_extendido', 'thal=7')}.",
         f"- Razón de verosimilitud básico vs. extendido: χ² = {f(E['LR_basico_vs_extendido']['LR_chi2'], 1)}, gl = {E['LR_basico_vs_extendido']['gl']}, p < 0,001.", "",
         "## Predicción (OE3–OE4, H-2 a H-4)", "",
         "| Modelo | AUC CV (P2,5–P97,5) | AUC fuera de muestra, DeLong IC 95 % | Brier | Pendiente de calibración | Sens./Esp. con umbral 0,5 | Esp. con sens. ≥ 90 % |",
         "|---|---|---|---|---|---|---|"]
    names = {"s1": "RL 4 continuas (semestre 1)", "basico": "RL básica (no invasiva)",
             "extendido": "RL extendida (+ imagen)", "arbol": "Árbol CART (extendido)", "rf": "Random Forest (extendido)"}
    for k, n in names.items():
        x = d[k]
        L.append(f"| {n} | {f(x['AUC_CV_media'])} ({f(x['AUC_CV_P2.5'])}–{f(x['AUC_CV_P97.5'])}) | "
                 f"{f(x['AUC_OOF_DeLong'])} ({f(x['AUC_OOF_IC_inf'])}–{f(x['AUC_OOF_IC_sup'])}) | {f(x['Brier_CV'])} | "
                 f"{f(x['pendiente_calibracion'], 2)} | {f(x['sens_0.5'])} / {f(x['esp_0.5'])} | {f(x['esp_u90'])} |")
    bo = P["bootstrap_optimismo"]
    L += ["",
          f"- **H-2:** AUC corregido por optimismo del modelo básico = {f(bo['basico']['auc_corregido'])} (optimismo {f(bo['basico']['optimismo'])}); límite inferior del IC de DeLong = {f(d['basico']['AUC_OOF_IC_inf'])} > 0,80 → **se rechaza H0**.",
          f"- **H-3:** ΔAUC extendido − básico = {f(nb('extendido_vs_basico')['diferencia_media'])} (IC 95 % {f(nb('extendido_vs_basico')['IC95_inf'])} a {f(nb('extendido_vs_basico')['IC95_sup'])}; p = {f(nb('extendido_vs_basico')['p'])}) → **la imagen aporta, aunque de forma modesta**.",
          f"- **H-4:** ΔAUC RL extendida − CART = {f(nb('extendido_vs_arbol')['diferencia_media'])} (IC 95 % {f(nb('extendido_vs_arbol')['IC95_inf'])} a {f(nb('extendido_vs_arbol')['IC95_sup'])}; p = {f(nb('extendido_vs_arbol')['p'])}) → **la RL discrimina mejor que el árbol**. RL extendida vs. RF: ΔAUC = {f(nb('extendido_vs_rf')['diferencia_media'])} (p = {f(nb('extendido_vs_rf')['p'])}), es decir, sin diferencia.",
          f"- Mejora frente al semestre 1: ΔAUC básico − 4 continuas = {f(nb('basico_vs_s1')['diferencia_media'])} (p < 0,001).",
          f"- Árbol: hiperparámetros modales {P['arbol_hiperparametros_modales']}, elegidos en solo el {f(100 * P['arbol_frecuencia_modal'], 0)} % de las particiones (inestable); pendiente de calibración {f(d['arbol']['pendiente_calibracion'], 2)} (sobreajuste).",
          f"- Tamizaje (RL básica, sensibilidad ≥ 90 %): umbral {f(d['basico']['umbral_sens90'], 2)}, especificidad {f(d['basico']['esp_u90'])}, VPN {f(d['basico']['VPN_u90'])}.", "",
          "## Tamaño de muestra", "",
          f"- Riley (2020): se requieren n = {T['basico']['n_requerido']} (básico) y n = {T['extendido']['n_requerido']} (extendido); se dispone de n = {D['n_completos']}. EPV = {f(E['basico']['epv'], 1)} (básico) y {f(E['extendido']['epv'], 1)} (extendido). Margen de la prevalencia: ± {f(100 * T['margen_prevalencia_alcanzado'], 1)} %.",
          ]
    (RESULTS / "RESUMEN_RESULTADOS.md").write_text("\n".join(L) + "\n", encoding="utf-8")

In [3]:
R = {}
for part in ["01_datos", "02_descriptivo", "03_asociacion", "04_prediccion"]:
    R.update(load_json(part))
(RESULTS / "resultados.json").write_text(
    json.dumps(R, ensure_ascii=False, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o)),
    encoding="utf-8")
write_summary(R)
display(Markdown((RESULTS / "RESUMEN_RESULTADOS.md").read_text(encoding="utf-8")))

# Resumen de resultados

> Generado automáticamente por `make all`. No editar a mano: toda cifra del paper y de la presentación debe salir de aquí o de `resultados.json`.

## Datos

- Fuente: UCI Machine Learning Repository, Heart Disease (Cleveland), DOI 10.24432/C52P4X. SHA-256 verificado.
- 303 registros; 6 con faltantes (ca: 4, thal: 2); muestra analítica n = 297, con 137 eventos (46,1 %).
- Auditoría Kaggle: target invertido = True; duplicados = 1; faltantes ocultos = {'ca==4': 5, 'thal==0': 2}.

## Asociación (OE2, H-1)

- Modelo básico: LR χ² = 174,7, gl = 14, p < 0,001 → **se rechaza H0 de H-1**. R² de Nagelkerke = 0,594; Hosmer-Lemeshow p = 0,532; VIF máximo = 4,19; Box-Tidwell p mínimo = 0,224.
- Variables significativas en el modelo básico: Sexo masculino; Tipo de dolor torácico: Asintomático (vs. angina típica); Frecuencia cardíaca máxima alcanzada (por 10 lpm); Depresión del ST inducida por ejercicio (por 1 mm).
- Variables significativas en el modelo extendido: Sexo masculino; Tipo de dolor torácico: Asintomático (vs. angina típica); Presión arterial sistólica en reposo (por 10 mmHg); Pendiente del ST en el pico de ejercicio: Plana (vs. ascendente); Vasos mayores coloreados por fluoroscopia (por cada vaso); Gammagrafía con talio: Defecto reversible (vs. normal).
- Dolor asintomático (básico): OR 10,60 (IC 95 % 3,17–35,40; p < 0,001). Sexo masculino: OR 7,48 (IC 95 % 3,25–17,20; p < 0,001).
- FC máxima por 10 lpm (básico): OR 0,82 (IC 95 % 0,68–1,00; p = 0,048). Depresión ST por mm: OR 1,82 (IC 95 % 1,23–2,69; p = 0,003).
- Colesterol por 40 mg/dL (básico): OR 1,21 (IC 95 % 0,92–1,58; p = 0,175). Edad por 10 años: OR 1,25 (IC 95 % 0,82–1,91; p = 0,294).
- Extendido: vasos por fluoroscopia OR 3,71 (IC 95 % 2,15–6,41; p < 0,001); talio con defecto reversible OR 4,03 (IC 95 % 1,75–9,27; p = 0,001).
- Razón de verosimilitud básico vs. extendido: χ² = 43,6, gl = 3, p < 0,001.

## Predicción (OE3–OE4, H-2 a H-4)

| Modelo | AUC CV (P2,5–P97,5) | AUC fuera de muestra, DeLong IC 95 % | Brier | Pendiente de calibración | Sens./Esp. con umbral 0,5 | Esp. con sens. ≥ 90 % |
|---|---|---|---|---|---|---|
| RL 4 continuas (semestre 1) | 0,749 (0,612–0,865) | 0,746 (0,690–0,801) | 0,205 | 0,93 | 0,650 / 0,769 | 0,344 |
| RL básica (no invasiva) | 0,874 (0,794–0,949) | 0,875 (0,836–0,914) | 0,146 | 0,91 | 0,752 / 0,844 | 0,613 |
| RL extendida (+ imagen) | 0,909 (0,829–0,964) | 0,904 (0,870–0,938) | 0,120 | 0,84 | 0,788 / 0,875 | 0,700 |
| Árbol CART (extendido) | 0,842 (0,715–0,940) | 0,847 (0,802–0,893) | 0,163 | 0,26 | 0,737 / 0,875 | 0,519 |
| Random Forest (extendido) | 0,910 (0,833–0,964) | 0,905 (0,872–0,938) | 0,127 | 1,59 | 0,759 / 0,875 | 0,719 |

- **H-2:** AUC corregido por optimismo del modelo básico = 0,880 (optimismo 0,021); límite inferior del IC de DeLong = 0,836 > 0,80 → **se rechaza H0**.
- **H-3:** ΔAUC extendido − básico = 0,035 (IC 95 % 0,008 a 0,062; p = 0,012) → **la imagen aporta, aunque de forma modesta**.
- **H-4:** ΔAUC RL extendida − CART = 0,067 (IC 95 % 0,021 a 0,113; p = 0,004) → **la RL discrimina mejor que el árbol**. RL extendida vs. RF: ΔAUC = -0,001 (p = 0,936), es decir, sin diferencia.
- Mejora frente al semestre 1: ΔAUC básico − 4 continuas = 0,125 (p < 0,001).
- Árbol: hiperparámetros modales {'max_depth': 4, 'min_samples_leaf': 10}, elegidos en solo el 32 % de las particiones (inestable); pendiente de calibración 0,26 (sobreajuste).
- Tamizaje (RL básica, sensibilidad ≥ 90 %): umbral 0,24, especificidad 0,613, VPN 0,883.

## Tamaño de muestra

- Riley (2020): se requieren n = 412 (básico) y n = 382 (extendido); se dispone de n = 297. EPV = 9,8 (básico) y 8,1 (extendido). Margen de la prevalencia: ± 5,7 %.
